# Uso de callbacks

En este ejercicio, implementaremos métodos de callbacks para detener el entrenamiento cuando se cumpla una métrica específica. Esta es una función útil, ya que permitirá no completar todas las épocas cuando se alcance este umbral. Por ejemplo, si configuramos 1000 épocas y la precisión deseada ya se alcanza en la época 200, el entrenamiento se detendrá automáticamente. Veamos cómo se implementa esto en las siguientes secciones.

## Cargamos y normalizamos el conjunto de datos MNIST de moda

Al igual que en el ejercicio anterior, volveremos a utilizar el conjunto de datos Fashion MNIST.Recuerda que debemos normalizar los valores de píxeles para ayudar a optimizar el entrenamiento.

In [18]:
import tensorflow as tf

from tensorflow.keras import layers, models

In [19]:
from tensorflow.keras.datasets import fashion_mnist


# Instanciamos el conjunto de datos
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()


# Cargamos el dataset

# Normalizamos los valores
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

print("Forma de X_train:", X_train.shape)
print("Forma de X_test:", X_test.shape)

Forma de X_train: (60000, 28, 28)
Forma de X_test: (10000, 28, 28)


## Creamos una clase callbacks

Puede crear un método callbacks definiendo una clase que herede la clase base [tf.keras.callbacks.Callback](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/Callback). Desde allí, pudemos definir los métodos disponibles para establecer dónde se ejecutará la evaluación del método. Por ejemplo, a continuación, utilizará el método [on_epoch_end()](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/Callback#on_epoch_end) para verificar el valor de la loss function al final de la época.

Crear una clase llamada myCallback que extienda tf.keras.callbacks.Callback.

Sobrescribir el método on_epoch_end(self, epoch, logs=None).

Comprobar en logs el valor de 'loss'.

Parar el entrenamiento cuando loss < 0.4.


In [20]:
class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        # Comprobamos el valor de la loss
        loss_actual = logs.get('loss')

        if loss_actual is not None and loss_actual < 0.4:
            print(f"\n Se detiene el entrenamiento: loss = {loss_actual:.4f} (menor que 0.4)")
            self.model.stop_training = True

## Definimos y compilamos el modelo

A continuación, definiremos y compilaremos el modelo. La arquitectura será similar a la que se construyó en el laboratorio anterior. Luego, estableceremos el optimizador, la loss function y las métricas que usaremos para el entrenamiento.

In [21]:
# Definimos el modelo
model = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),       # Aplanamos la imagen 28x28
    layers.Dense(128, activation='relu'),       # Capa oculta
    layers.Dense(10, activation='softmax')      # Capa de salida (10 clases)
])

# Compilamos el modelo
model.compile(
    optimizer='adam',                          # Optimizador
    loss='sparse_categorical_crossentropy',    # Pérdida
    metrics=['accuracy']                       # Métricas
)

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

### Entrenamos al modelo

Ahora estamos listos para entrenar el modelo. Para configurar la devolución de llamada, simplemente configure el parámetro `callbacks` en la instancia `myCallback` que declaramos anteriormente.

In [22]:
# Creamos la instancia del callback
callbacks = myCallback()

# Entrenamos el modelo
history = model.fit(
    X_train, y_train,
    epochs=10,
    validation_data=(X_test, y_test),
    callbacks=[callbacks]
)

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.7805 - loss: 0.6260 - val_accuracy: 0.8503 - val_loss: 0.4247
Epoch 2/10
1873/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8602 - loss: 0.3873
 Se detiene el entrenamiento: loss = 0.3755 (menor que 0.4)
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8602 - loss: 0.3873 - val_accuracy: 0.8566 - val_loss: 0.3937


Notarás que el entrenamiento no necesita completar las 10 épocas. Al evaluar el méotodo al final de la época, podemos verificar los parámetros de entrenamiento y comparar si cumplimos con el umbral establecido en la definición de la función. En este caso, simplemente se detendrá cuando la loss function caiga por debajo de `0.40` después de la época actual.

*Desafío opcional: modifica el código para que el entrenamiento se detenga cuando la métrica de precisión supere el 60 %.*

In [27]:
class myNewCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        # Comprobamos el valor de la loss
        accuracy_actual = logs.get('accuracy')

        if accuracy_actual is not None and accuracy_actual > 0.6:
            print(f"\n Se detiene el entrenamiento: accuaracy = {accuracy_actual:.4f} (mayor al 60%)")
            self.model.stop_training = True

In [28]:
# Definimos el modelo
model = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),       # Aplanamos la imagen 28x28
    layers.Dense(128, activation='relu'),       # Capa oculta
    layers.Dense(10, activation='softmax')      # Capa de salida (10 clases)
])

# Compilamos el modelo
model.compile(
    optimizer='adam',                          # Optimizador
    loss='sparse_categorical_crossentropy',    # Pérdida
    metrics=['accuracy']                       # Métricas
)

model.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_3 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
# Creamos la nueva instancia del callback
myNewCallback = myNewCallback()

# Entrenamos el modelo
history = model.fit(
    X_train, y_train,
    epochs=10,
    validation_data=(X_test, y_test),
    callbacks=[myNewCallback]
)

Epoch 1/10
1872/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7808 - loss: 0.6340
 Se detiene el entrenamiento: accuaracy = 0.8249 (mayor al 60%)
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.7809 - loss: 0.6337 - val_accuracy: 0.8500 - val_loss: 0.4166
